In [1]:
import numpy as np

In [2]:
def get_primes(nmax):
    primes = [2]
    while primes[-1] < nmax:
        primes += [primes[-1]+1]
        while any([primes[-1]%i==0 for i in primes[:-1]]):
            primes[-1] += 1
    return np.array(primes if primes[-1]==nmax else primes[:-1])

def get_quads(primes, target):
    primes = np.array(primes, dtype=int)
    quads = np.zeros((1, 1), dtype=int)
    for i in range(3):
        filt = (((nc:=np.tile(primes, quads.shape[0])) > np.repeat(quads[:, -1], primes.shape[0]))
                & (np.repeat(np.sum(quads, axis=1), primes.shape[0]) + nc <= 2026))
        quads = np.column_stack((np.repeat(quads[:, -i:], primes.shape[0], axis=0)[filt], nc[filt]))
    filt = ((nc:=target - np.sum(quads, axis=1)) > quads[:, -1]) & np.isin(nc, primes)
    return np.column_stack((quads[filt], nc[filt]))

def get_perm_ix(n):
    ix = -np.ones(([1,1]), dtype=int)
    for i in range(n):
        filt = np.all((nc:=np.tile(np.expand_dims(np.arange(n), 1), (ix.shape[0], 1))) 
                      != (ec:=np.repeat(ix, n, axis=0)), 
                      axis=1)
        ix = np.column_stack((ec[:, -i:], nc))[filt]
    return ix
PERM4IX = get_perm_ix(4)

def find_ms(prime_quads, target, grid=None):
    if grid is None:
        grid = np.empty((0, 4), dtype=int)
    if grid.shape[0] == 3:
        return (np.append(grid, [z], axis=0)
                if np.all(~np.isin((z:=target - np.sum(grid, axis=0)), grid))
                   and np.any(np.all(prime_quads == [np.sort(z)], axis=1))
                   and z[-1]+np.sum(np.identity(4, dtype=int)[:3]*grid) 
                       == z[0]+np.sum(np.flip(np.identity(4, dtype=int), axis=1)[:3]*grid) 
                       == target
                else None)
    else:
        for i in prime_quads:
            if np.all(~np.isin(i, grid)):
                for j in i[PERM4IX]:
                    if (z:=find_ms(prime_quads, 
                                   target, 
                                   np.append(grid, [j], axis=0))) is not None:
                        return z
        return "NO MAGIC SQUARE POSSIBLE"

In [3]:
target = 2026
primes = get_primes(target)
prime_quads = get_quads(primes, target)
# find_ms(prime_quads, target)

In [58]:
answer = np.array([23, 479, 461, 1063, 1093, 431, 389, 113, 419, 83, 1123, 401, 491, 1033, 53, 449]).reshape(4,4)
answer

array([[  23,  479,  461, 1063],
       [1093,  431,  389,  113],
       [ 419,   83, 1123,  401],
       [ 491, 1033,   53,  449]])

In [59]:
grid = np.empty((0, 4), dtype=int)

In [60]:
grid_poss = [np.append(grid, [j], axis=0)
             for i in prime_quads if np.all(~np.isin(i, grid))
             for j in i[PERM4IX]]

In [ ]:
if (z:=np.arange(len(grid_poss))[np.all(np.stack(grid_poss) == [answer[:grid_poss[0].shape[0]]], axis=(1,2))]).shape[0]:
    ix = z[0].item()
ix

495602

In [64]:
grid = grid_poss[ix]
grid

array([[  23,  479,  461, 1063]])

In [65]:
grid_poss = [np.append(grid, [j], axis=0)
             for i in prime_quads if np.all(~np.isin(i, grid))
             for j in i[PERM4IX]]

In [ ]:
if (z:=np.arange(len(grid_poss))[np.all(np.stack(grid_poss) == [answer[:grid_poss[0].shape[0]]], axis=(1,2))]).shape[0]:
    ix = z[0].item()
ix

1285703

In [74]:
grid = grid_poss[ix]
grid

array([[  23,  479,  461, 1063],
       [1093,  431,  389,  113]])

In [75]:
grid_poss = [np.append(grid, [j], axis=0)
             for i in prime_quads if np.all(~np.isin(i, grid))
             for j in i[PERM4IX]]

In [79]:
if (z:=np.arange(len(grid_poss))[np.all(np.stack(grid_poss) == [answer[:grid_poss[0].shape[0]]], axis=(1,2))]).shape[0]:
    ix = z[0].item()
ix

968029

In [80]:
grid = grid_poss[ix]
grid

array([[  23,  479,  461, 1063],
       [1093,  431,  389,  113],
       [ 419,   83, 1123,  401]])

In [86]:
np.all(np.isin((z:=target - np.sum(grid, axis=0)), primes)), np.sum(z) == target

(np.True_, np.True_)

In [90]:
np.sum(grid[[0,1,2],[0,1,2]])+z[3] == target, np.sum(grid[[0,1,2],[3,2,1]])+z[0] == target

(np.True_, np.True_)